In [ ]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 18, Derivator identities (§3.1.5)

Companion notebook to [18_derivator_identities.md](18_derivator_identities.md). The six §3.1.5 identities equate Cartan / SN derivators to sums of Cartan-remainder corrections (`K_V` / `K̃_η`). All six close mechanically through `KoszulProblem.prove_derivator`.

## Setup, `KoszulProblem` as the entry point

`KoszulProblem` carries the bivector, form / multivector inventories, and exposes pre-bundled `derivator_form_engine` / `derivator_multivector_engine` accessors. `assume_poisson()` flags `π` as `Poisson`, that's what unlocks `d̃² V → 0` and the SN-bracket Jacobi inside the multivector engine.

In [ ]:
from jacopy.algebra.derivation import Act, Derivation
from jacopy.brackets.base import BracketApply
from jacopy.brackets.schouten import sn as default_sn
from jacopy.calculus.cartan_remainder import K
from jacopy.calculus.derivator import derivator
from jacopy.calculus.exterior_d import d as default_d
from jacopy.calculus.interior import interior
from jacopy.calculus.lie_derivative import lie_derivative
from jacopy.calculus.tilde import (
    K_tilde, tilde_d, tilde_interior, tilde_lie,
)
from jacopy.core.expr import Neg, Sum, Symbol
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.library.koszul_problem import KoszulProblem

reg = PropertyRegistry()
pi = Symbol("π")
omega, eta, mu = (Symbol(s) for s in ("ω", "η", "μ"))
U, V, W = (Symbol(s) for s in ("U", "V", "W"))
for f in (omega, eta, mu, U, V, W):
    reg.declare(f, Graded(degree=1))

Y = Derivation("Y", 0)
xi = Symbol("ξ"); reg.declare(xi, Graded(degree=1))

prob = KoszulProblem(
    pi, (omega, eta, mu),
    registry=reg,
    multivectors=((U, 1), (V, 1), (W, 1)),
)
prob.assume_poisson()
K_b = prob.koszul_bracket

print(f"form engine        : {len(prob.derivator_form_engine().definitions)} rules")
print(f"multivector engine : {len(prob.derivator_multivector_engine().definitions)} rules")

## Form-side identity (1)

$$D^{T^*M}_{L_U}(\eta, \mu) = L_{\tilde K_\eta U} \mu + K_{\tilde K_\mu U} \eta$$

The cyclic identity, three nested Koszul-bracket expansions feed into the obstruction. The 109 steps cover Cartan-magic expansions, operator-commutator folds, and `K̃_η U → −L̃_η U + d̃ ι̃_η U` polarity flips.

In [ ]:
lhs = derivator(lie_derivative(U), K_b, eta, mu)

K_tilde_eta_U = Act(K_tilde(eta, pi), U)
K_tilde_mu_U  = Act(K_tilde(mu, pi),  U)
rhs = Sum(
    Act(lie_derivative(K_tilde_eta_U), mu),
    Act(K(K_tilde_mu_U),               eta),
)

chain = prob.prove_derivator(lhs, rhs, eval_args=(Y,), side="form")
print(f"(1) form-side closes in {len(chain)} steps")

## Dual multivector-side identity (1')

$$\tilde D^{SN}_{\tilde L_\eta}(U, V) = \tilde L_{K_U \eta} V + \tilde K_{K_V \eta} U$$

`side="multivector"` routes through `derivator_multivector_engine` and uses `slot_kind="covector"` for the `MultiEval` wrap, same discipline as `prove_tilde_cartan_relation` (tutorial 17).

In [ ]:
lhs = derivator(tilde_lie(eta, pi), default_sn, U, V)

K_U_eta = Act(K(U), eta)
K_V_eta = Act(K(V), eta)
rhs = Sum(
    Act(tilde_lie(K_U_eta, pi), V),
    Act(K_tilde(K_V_eta, pi),   U),
)

chain = prob.prove_derivator(
    lhs, rhs, eval_args=(xi,), side="multivector",
)
print(f"(1') multivector-side closes in {len(chain)} steps")

## The full table

All six identities close in one `prob.prove_derivator(...)` call. Step counts:

| # | side | steps |
|---|---|---|
| (1) `D^{T*M}_{L_U}(η, μ) = L_{K̃_η U} μ + K_{K̃_μ U} η` | form | 109 |
| (2) `L_{d̃ ι̃_η U} μ = −[d ι_U η, μ]_K` | form | 21 |
| (3) `0 = d ι_{L̃_ω W} η − d ι_{d̃ ι̃_η W} ω + d ι_W [ω, η]_K` | form | 30 |
| (1') `D̃^{SN}_{L̃_η}(U, V) = L̃_{K_U η} V + K̃_{K_V η} U` | multivec | 117 |
| (2') `L̃_{d ι_U η} V = −[d̃ ι̃_η U, V]_SN` | multivec | 25 |
| (3') `0 = d̃ ι̃_{L_U η} V − d̃ ι̃_{d ι_V η} U + d̃ ι̃_η [U, V]_SN` | multivec | 23 |

The cyclic (1)/(1') are the heaviest because three nested bracket expansions feed the obstruction. (2)/(2') and (3)/(3') skip the cycle and run lean.

## When you'd reach past the wrapper

Use `prove_derivator_identity` from `jacopy.calculus.derivator` directly when:

* you need a custom rule layered on (`derivator_form_engine()` returns a fresh engine to extend);
* the bracket isn't Koszul or SN (write your own engine factory mirroring the `derivator_form_engine` pattern);
* you're proving an identity outside §3.1.5's derivator shape (Cartan magic, `d² = 0`, those belong to tutorial 15's `intrinsic_engine`).

## Summary

* Derivator `D^E_φ(u, v) := φ[u,v]_E − [φu, v]_E − [u, φv]_E` measures `φ`'s failure to be a graded derivation of `[·, ·]_E`.
* Six §3.1.5 identities pin Cartan / SN derivators to sums of `K_V` / `K̃_η` corrections, three form-side, three dual.
* `KoszulProblem.prove_derivator(lhs, rhs, *, eval_args, side)` closes all six in one call: `side="form"` for (1)/(2)/(3), `side="multivector"` for (1')/(2')/(3').
* Step counts 109/21/30 (form) and 117/25/23 (multivec), (1)/(1') are heaviest because three nested bracket expansions feed the obstruction.
* `prove_derivator_identity` is the engine-level entry point when you've assembled your own engine.